# Sensitivity Analysis: Outcome-Adjacent Predictors

This notebook performs a separate sensitivity analysis of the corrected leakage-safe 5-fold CGPA classification experiment.

**Primary purpose:** compare the full predictor set with a reduced predictor set after removing five potentially outcome-adjacent variables:

- `Class_Attendance`
- `Sleepiness_During_Class`
- `Skip_Class_for_Sleep`
- `Focus_on_Academic_Task`
- `Impact_of_Sleep_on_Academic`

The original corrected 5-fold notebook is not modified by this experiment.

The target remains `CGPA3_Class`.

The same stratified 5-fold assignments, model configurations, CTGAN configuration, and evaluation metrics are used for both feature sets. CTGAN is fitted separately within each fold using only the fold-training data, and validation data remain real and untouched.


## 1. Imports and Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

TARGET_COLUMN = 'CGPA3_Class'

REMOVED_FEATURES = [
    'Class_Attendance',
    'Sleepiness_During_Class',
    'Skip_Class_for_Sleep',
    'Focus_on_Academic_Task',
    'Impact_of_Sleep_on_Academic'
]

EXCLUDED_FROM_MODEL = ['CGPA3_Class', 'Target', 'Current_CGPA5']

CTGAN_CONFIG = {
    'epochs': 500,
    'batch_size': 50,
    'generator_dim': (256, 256),
    'discriminator_dim': (256, 256),
    'generator_lr': 2e-4,
    'discriminator_lr': 2e-4
}

N_FOLDS = 5
RANDOM_STATE = 42


Using device: cuda


## 2. Load the Original Real Dataset

The sensitivity analysis starts from the same original real dataset used by the corrected 5-fold analysis. No pre-generated synthetic dataset is used.


In [2]:
real_df = pd.read_csv(r'../data/Final_Encoded.csv')

print(f"Dataset shape: {real_df.shape}")
print(f"Target: {TARGET_COLUMN}")
print(f"Target values: {sorted(real_df[TARGET_COLUMN].unique().tolist())}")
print(f"Class counts: {real_df[TARGET_COLUMN].value_counts().sort_index().to_dict()}")

full_feature_columns = [
    col for col in real_df.columns
    if col not in EXCLUDED_FROM_MODEL
]

reduced_feature_columns = [
    col for col in full_feature_columns
    if col not in REMOVED_FEATURES
]

print(f"Full feature count: {len(full_feature_columns)}")
print(f"Reduced feature count: {len(reduced_feature_columns)}")
print(f"Removed features: {REMOVED_FEATURES}")


Dataset shape: (1481, 31)
Target: CGPA3_Class
Target values: [0, 1, 2]
Class counts: {0: 655, 1: 608, 2: 218}
Full feature count: 29
Reduced feature count: 24
Removed features: ['Class_Attendance', 'Sleepiness_During_Class', 'Skip_Class_for_Sleep', 'Focus_on_Academic_Task', 'Impact_of_Sleep_on_Academic']


In [3]:
# Verification of the two feature sets before any model fitting.
assert TARGET_COLUMN not in full_feature_columns
assert TARGET_COLUMN not in reduced_feature_columns
assert 'Target' not in full_feature_columns
assert 'Target' not in reduced_feature_columns
assert 'Current_CGPA5' not in full_feature_columns
assert 'Current_CGPA5' not in reduced_feature_columns

assert all(feature in full_feature_columns for feature in REMOVED_FEATURES)
assert not any(feature in reduced_feature_columns for feature in REMOVED_FEATURES)
assert len(full_feature_columns) - len(reduced_feature_columns) == 5

print("Feature-set verification passed.")


Feature-set verification passed.


## 3. Define the Same Five Stratified Folds Once

The fold assignments are created once from the original real dataset and reused for both the Full and Reduced feature sets. This makes the sensitivity comparison use identical train/validation partitions.


In [4]:
cv_splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_splits = list(
    cv_splitter.split(real_df, real_df[TARGET_COLUMN])
)

print(f"Created {len(fold_splits)} identical stratified folds for both feature sets.")

for fold_number, (train_idx, validation_idx) in enumerate(fold_splits, start=1):
    print(
        f"Fold {fold_number}: "
        f"real train={len(train_idx)}, "
        f"real validation={len(validation_idx)}"
    )


Created 5 identical stratified folds for both feature sets.
Fold 1: real train=1184, real validation=297
Fold 2: real train=1185, real validation=296
Fold 3: real train=1185, real validation=296
Fold 4: real train=1185, real validation=296
Fold 5: real train=1185, real validation=296


## 4. Neural Network Architecture

This is the same neural-network architecture used in the corrected 5-fold notebook: hidden layers of 128, 64, and 32 units, ReLU activations, 0.3 dropout, and a three-class output.


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes):
        super(NeuralNetwork, self).__init__()

        layers = []
        prev_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


## 5. Leakage-Safe Sensitivity Analysis

For every feature set and every fold:

1. Split real observations using the precomputed stratified fold.
2. Fit normalization using fold-training predictors only.
3. Select the minority class from fold-training data.
4. Fit a fresh CTGAN only on the fold-training minority data.
5. Generate synthetic minority observations for training only.
6. Train the six classifiers on the augmented training set.
7. Evaluate only on the untouched real validation fold.

The Reduced feature set excludes exactly the five reviewer-identified variables from both model inputs and the CTGAN training schema.


In [6]:
sensitivity_fold_results = []

feature_sets = {
    'Full': full_feature_columns,
    'Reduced': reduced_feature_columns
}

for feature_set_name, feature_columns in feature_sets.items():

    print("\n" + "=" * 90)
    print(f"FEATURE SET: {feature_set_name}")
    print("=" * 90)

    for fold_number, (train_indices, validation_indices) in enumerate(
        fold_splits, start=1
    ):
        fold_real_train = real_df.iloc[train_indices].copy()
        fold_real_validation = real_df.iloc[validation_indices].copy()

        # Normalize only the predictors actually used by this feature set.
        fold_train_max_values = fold_real_train[feature_columns].max()

        fold_train_normalized = fold_real_train[feature_columns].copy()
        fold_validation_normalized = fold_real_validation[feature_columns].copy()

        for column in feature_columns:
            if fold_train_max_values[column] > 0:
                fold_train_normalized[column] = (
                    fold_train_normalized[column] / fold_train_max_values[column]
                )
                fold_validation_normalized[column] = (
                    fold_validation_normalized[column] / fold_train_max_values[column]
                )

        fold_train_normalized[TARGET_COLUMN] = (
            fold_real_train[TARGET_COLUMN].astype(int).values
        )
        fold_validation_normalized[TARGET_COLUMN] = (
            fold_real_validation[TARGET_COLUMN].astype(int).values
        )

        # Determine the minority class using fold-training labels only.
        fold_class_counts = fold_train_normalized[TARGET_COLUMN].value_counts()
        fold_minority_class = fold_class_counts.idxmin()
        fold_minority = fold_train_normalized[
            fold_train_normalized[TARGET_COLUMN] == fold_minority_class
        ].copy()

        # CTGAN is trained only on the current fold's real minority training data.
        fold_ctgan_columns = feature_columns + [TARGET_COLUMN]
        fold_minority_ctgan = fold_minority[fold_ctgan_columns].copy()

        fold_metadata = SingleTableMetadata()
        fold_metadata.detect_from_dataframe(fold_minority_ctgan)

        for column in fold_minority_ctgan.columns:
            fold_metadata.update_column(column, sdtype='numerical')

        fold_synthesizer = CTGANSynthesizer(
            fold_metadata,
            epochs=CTGAN_CONFIG['epochs'],
            batch_size=CTGAN_CONFIG['batch_size'],
            generator_dim=CTGAN_CONFIG['generator_dim'],
            discriminator_dim=CTGAN_CONFIG['discriminator_dim'],
            generator_lr=CTGAN_CONFIG['generator_lr'],
            discriminator_lr=CTGAN_CONFIG['discriminator_lr'],
            verbose=False
        )

        fold_synthesizer.fit(fold_minority_ctgan)

        fold_majority_count = fold_train_normalized[TARGET_COLUMN].value_counts().max()
        fold_synthetic_count = int(
            fold_majority_count - len(fold_minority)
        )

        fold_synthetic = fold_synthesizer.sample(
            num_rows=fold_synthetic_count
        )

        for column in feature_columns:
            fold_synthetic[column] = fold_synthetic[column].clip(0, 1)

        fold_synthetic[TARGET_COLUMN] = int(fold_minority_class)

        fold_augmented_train = pd.concat(
            [
                fold_train_normalized[feature_columns + [TARGET_COLUMN]],
                fold_synthetic[feature_columns + [TARGET_COLUMN]]
            ],
            ignore_index=True
        )

        X_fold_train = fold_augmented_train[feature_columns].values
        y_fold_train = fold_augmented_train[TARGET_COLUMN].astype(int).values

        X_fold_validation = fold_validation_normalized[feature_columns].values
        y_fold_validation = fold_validation_normalized[TARGET_COLUMN].astype(int).values

        fold_models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                min_samples_split=5,
                min_samples_leaf=3,
                subsample=0.8,
                max_features='sqrt',
                validation_fraction=0.1,
                n_iter_no_change=20,
                random_state=42
            ),
            'XGBoost': XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                min_child_weight=3,
                subsample=0.8,
                colsample_bytree=0.8,
                gamma=1,
                random_state=42,
                use_label_encoder=False,
                eval_metric='mlogloss'
            ),
            'CatBoost': CatBoostClassifier(
                iterations=100,
                depth=5,
                learning_rate=0.1,
                random_state=42,
                verbose=False
            ),
            'Logistic Regression': LogisticRegression(
                max_iter=2000,
                C=0.5,
                penalty='elasticnet',
                solver='saga',
                l1_ratio=0.5,
                multi_class='multinomial',
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
        }

        for model_name, fold_model in fold_models.items():
            fold_model.fit(X_fold_train, y_fold_train)

            fold_predictions = fold_model.predict(X_fold_validation)
            fold_probabilities = fold_model.predict_proba(X_fold_validation)

            sensitivity_fold_results.append({
                'Feature_Set': feature_set_name,
                'Fold': fold_number,
                'Model': model_name,
                'Accuracy': accuracy_score(
                    y_fold_validation, fold_predictions
                ),
                'Precision': precision_score(
                    y_fold_validation, fold_predictions, average='macro'
                ),
                'Recall': recall_score(
                    y_fold_validation, fold_predictions, average='macro'
                ),
                'F1': f1_score(
                    y_fold_validation, fold_predictions, average='macro'
                ),
                'ROC-AUC': roc_auc_score(
                    y_fold_validation,
                    fold_probabilities,
                    multi_class='ovr',
                    average='macro'
                )
            })

        # Same neural-network configuration as the corrected 5-fold notebook.
        fold_ann = NeuralNetwork(
            X_fold_train.shape[1],
            [128, 64, 32],
            3
        ).to(device)

        fold_ann_criterion = nn.CrossEntropyLoss()
        fold_ann_optimizer = optim.Adam(
            fold_ann.parameters(),
            lr=0.001
        )

        fold_train_dataset = TensorDataset(
            torch.FloatTensor(X_fold_train).to(device),
            torch.LongTensor(y_fold_train).to(device)
        )

        fold_train_loader = DataLoader(
            fold_train_dataset,
            batch_size=32,
            shuffle=True
        )

        fold_ann.train()

        for epoch in range(100):
            for batch_X, batch_y in fold_train_loader:
                fold_ann_optimizer.zero_grad()
                fold_outputs = fold_ann(batch_X)
                fold_loss = fold_ann_criterion(fold_outputs, batch_y)
                fold_loss.backward()
                fold_ann_optimizer.step()

        fold_ann.eval()

        with torch.no_grad():
            validation_tensor = torch.FloatTensor(
                X_fold_validation
            ).to(device)

            fold_ann_outputs = fold_ann(validation_tensor)

            fold_ann_predictions = torch.argmax(
                fold_ann_outputs, dim=1
            ).cpu().numpy()

            fold_ann_probabilities = torch.softmax(
                fold_ann_outputs, dim=1
            ).cpu().numpy()

        sensitivity_fold_results.append({
            'Feature_Set': feature_set_name,
            'Fold': fold_number,
            'Model': 'Neural Network',
            'Accuracy': accuracy_score(
                y_fold_validation, fold_ann_predictions
            ),
            'Precision': precision_score(
                y_fold_validation, fold_ann_predictions, average='macro'
            ),
            'Recall': recall_score(
                y_fold_validation, fold_ann_predictions, average='macro'
            ),
            'F1': f1_score(
                y_fold_validation, fold_ann_predictions, average='macro'
            ),
            'ROC-AUC': roc_auc_score(
                y_fold_validation,
                fold_ann_probabilities,
                multi_class='ovr',
                average='macro'
            )
        })

        print(
            f"{feature_set_name} | Fold {fold_number}: "
            f"train={len(fold_real_train)}, "
            f"validation={len(fold_real_validation)}, "
            f"minority={len(fold_minority)}, "
            f"synthetic={fold_synthetic_count}, "
            f"augmented={len(fold_augmented_train)}"
        )

print("\nSensitivity analysis model fitting completed.")



FEATURE SET: Full
Full | Fold 1: train=1184, validation=297, minority=174, synthetic=350, augmented=1534
Full | Fold 2: train=1185, validation=296, minority=175, synthetic=349, augmented=1534
Full | Fold 3: train=1185, validation=296, minority=175, synthetic=349, augmented=1534
Full | Fold 4: train=1185, validation=296, minority=174, synthetic=350, augmented=1535
Full | Fold 5: train=1185, validation=296, minority=174, synthetic=350, augmented=1535

FEATURE SET: Reduced
Reduced | Fold 1: train=1184, validation=297, minority=174, synthetic=350, augmented=1534
Reduced | Fold 2: train=1185, validation=296, minority=175, synthetic=349, augmented=1534
Reduced | Fold 3: train=1185, validation=296, minority=175, synthetic=349, augmented=1534
Reduced | Fold 4: train=1185, validation=296, minority=174, synthetic=350, augmented=1535
Reduced | Fold 5: train=1185, validation=296, minority=174, synthetic=350, augmented=1535

Sensitivity analysis model fitting completed.


## 6. Fold-Level Results and Summary

The expected fold-level result count is:

**2 feature sets × 5 folds × 6 models = 60 rows**

The summary contains:

**2 feature sets × 6 models = 12 rows**


In [7]:
sensitivity_fold_results_df = pd.DataFrame(
    sensitivity_fold_results
).sort_values(
    ['Feature_Set', 'Model', 'Fold']
).reset_index(drop=True)

sensitivity_summary_df = (
    sensitivity_fold_results_df
    .groupby(['Feature_Set', 'Model'])
    .agg(
        Accuracy_mean=('Accuracy', 'mean'),
        Accuracy_std=('Accuracy', 'std'),
        Precision_mean=('Precision', 'mean'),
        Precision_std=('Precision', 'std'),
        Recall_mean=('Recall', 'mean'),
        Recall_std=('Recall', 'std'),
        F1_mean=('F1', 'mean'),
        F1_std=('F1', 'std'),
        ROC_AUC_mean=('ROC-AUC', 'mean'),
        ROC_AUC_std=('ROC-AUC', 'std')
    )
    .reset_index()
)

print("Fold-level sensitivity results:")
display(sensitivity_fold_results_df)

print("\nMean ± SD sensitivity results:")
display(sensitivity_summary_df)

print(
    f"\nFold-level rows: {len(sensitivity_fold_results_df)} "
    f"(expected 60)"
)
print(
    f"Summary rows: {len(sensitivity_summary_df)} "
    f"(expected 12)"
)


Fold-level sensitivity results:


,Feature_Set,Fold,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Full,1,CatBoost,0.508418,0.515979,0.509515,0.510770,0.655719
1,Full,2,CatBoost,0.459459,0.449329,0.449363,0.449035,0.624310
2,Full,3,CatBoost,0.466216,0.435779,0.443661,0.437498,0.631340
3,Full,4,CatBoost,0.456081,0.481214,0.429705,0.446210,0.640414
4,Full,5,CatBoost,0.452703,0.447526,0.446023,0.445618,0.621825
5,Full,1,Gradient Boosting,0.494949,0.536447,0.480714,0.500616,0.655014
6,Full,2,Gradient Boosting,0.479730,0.509814,0.456280,0.474315,0.615851
7,Full,3,Gradient Boosting,0.503378,0.477631,0.478736,0.477852,0.633940
8,Full,4,Gradient Boosting,0.500000,0.524842,0.463415,0.480983,0.648126
9,Full,5,Gradient Boosting,0.516892,0.543629,0.502555,0.518314,0.634062



Mean ± SD sensitivity results:


,Feature_Set,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_mean,F1_std,ROC_AUC_mean,ROC_AUC_std
0,Full,CatBoost,0.468575,0.022826,0.465966,0.032642,0.455654,0.031026,0.457826,0.029906,0.634722,0.013779
1,Full,Gradient Boosting,0.498990,0.013491,0.518473,0.026170,0.476340,0.017900,0.490416,0.018631,0.637399,0.015106
2,Full,Logistic Regression,0.424713,0.025772,0.410780,0.020843,0.436296,0.023976,0.412735,0.022362,0.603941,0.024045
3,Full,Neural Network,0.455080,0.018122,0.444534,0.023636,0.451688,0.026923,0.443991,0.023083,0.618732,0.025508
4,Full,Random Forest,0.481411,0.027642,0.503635,0.049521,0.464273,0.033563,0.476052,0.038895,0.633000,0.020570
5,Full,XGBoost,0.481434,0.016021,0.508287,0.032093,0.459946,0.014092,0.475033,0.013671,0.634383,0.017809
6,Reduced,CatBoost,0.453058,0.014969,0.456513,0.033325,0.431158,0.028141,0.438591,0.025562,0.611710,0.027066
7,Reduced,Gradient Boosting,0.466573,0.010314,0.474616,0.023239,0.435078,0.022324,0.446237,0.019989,0.613381,0.020780
8,Reduced,Logistic Regression,0.401112,0.034809,0.383865,0.030107,0.403341,0.027214,0.385491,0.029582,0.585048,0.017053
9,Reduced,Neural Network,0.449704,0.028555,0.426522,0.025555,0.429873,0.035606,0.422020,0.028162,0.594061,0.034052



Fold-level rows: 60 (expected 60)
Summary rows: 12 (expected 12)


In [8]:
# Compare Reduced minus Full for each model and metric.
full_summary = sensitivity_summary_df[
    sensitivity_summary_df['Feature_Set'] == 'Full'
].set_index('Model')

reduced_summary = sensitivity_summary_df[
    sensitivity_summary_df['Feature_Set'] == 'Reduced'
].set_index('Model')

comparison_rows = []

for model_name in full_summary.index:
    row = {'Model': model_name}

    for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']:
        row[f'{metric}_Reduced_minus_Full'] = (
            reduced_summary.loc[model_name, f'{metric}_mean']
            - full_summary.loc[model_name, f'{metric}_mean']
        )

    comparison_rows.append(row)

sensitivity_difference_df = pd.DataFrame(comparison_rows)

print("Descriptive difference: Reduced minus Full")
display(sensitivity_difference_df)


Descriptive difference: Reduced minus Full


,Model,Accuracy_Reduced_minus_Full,Precision_Reduced_minus_Full,Recall_Reduced_minus_Full,F1_Reduced_minus_Full,ROC_AUC_Reduced_minus_Full
0,CatBoost,-0.015518,-0.009453,-0.024495,-0.019235,-0.023012
1,Gradient Boosting,-0.032417,-0.043857,-0.041262,-0.044179,-0.024018
2,Logistic Regression,-0.023601,-0.026915,-0.032955,-0.027244,-0.018893
3,Neural Network,-0.005376,-0.018012,-0.021815,-0.021972,-0.024671
4,Random Forest,-0.040506,-0.049429,-0.057364,-0.057475,-0.024299
5,XGBoost,-0.021615,-0.037511,-0.034368,-0.035888,-0.027425


## 7. Save New Sensitivity-Analysis Results

These are new output files. Existing result files are not overwritten.


In [9]:
FOLD_RESULTS_PATH = '../results/sensitivity_5fold_fold_results.csv'
SUMMARY_RESULTS_PATH = '../results/sensitivity_5fold_model_comparison.csv'
DIFFERENCE_RESULTS_PATH = '../results/sensitivity_5fold_reduced_minus_full.csv'

sensitivity_fold_results_df.to_csv(
    FOLD_RESULTS_PATH,
    index=False
)

sensitivity_summary_df.to_csv(
    SUMMARY_RESULTS_PATH,
    index=False
)

sensitivity_difference_df.to_csv(
    DIFFERENCE_RESULTS_PATH,
    index=False
)

print(f"Saved fold-level results to: {FOLD_RESULTS_PATH}")
print(f"Saved summary results to: {SUMMARY_RESULTS_PATH}")
print(f"Saved Reduced-minus-Full comparison to: {DIFFERENCE_RESULTS_PATH}")


Saved fold-level results to: ../results/sensitivity_5fold_fold_results.csv
Saved summary results to: ../results/sensitivity_5fold_model_comparison.csv
Saved Reduced-minus-Full comparison to: ../results/sensitivity_5fold_reduced_minus_full.csv


## 8. Final Verification

This cell checks the required structural conditions before the results are used in the manuscript or reviewer response.


In [10]:
# Required result counts.
assert len(sensitivity_fold_results_df) == 60
assert len(sensitivity_summary_df) == 12

# Required feature removal.
assert set(full_feature_columns) - set(reduced_feature_columns) == set(REMOVED_FEATURES)
assert set(reduced_feature_columns) == (
    set(full_feature_columns) - set(REMOVED_FEATURES)
)

# Target remains unchanged.
assert TARGET_COLUMN == 'CGPA3_Class'

# Required models.
expected_models = {
    'Random Forest',
    'Gradient Boosting',
    'XGBoost',
    'CatBoost',
    'Logistic Regression',
    'Neural Network'
}

assert set(sensitivity_fold_results_df['Model']) == expected_models
assert set(sensitivity_summary_df['Model']) == expected_models

# Required feature sets.
assert set(sensitivity_fold_results_df['Feature_Set']) == {'Full', 'Reduced'}
assert set(sensitivity_summary_df['Feature_Set']) == {'Full', 'Reduced'}

# Every feature set/model has five folds.
counts = (
    sensitivity_fold_results_df
    .groupby(['Feature_Set', 'Model'])
    .size()
)

assert (counts == 5).all()

# No missing metrics.
metric_columns = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1',
    'ROC-AUC'
]

assert not sensitivity_fold_results_df[metric_columns].isna().any().any()

print("FINAL VERIFICATION PASSED")
print("- 60 fold-level results")
print("- 12 summary rows")
print("- exactly 5 removed variables in Reduced feature set")
print("- 6 models × 5 folds × 2 feature sets")
print("- no missing evaluation metrics")
print("- same precomputed fold assignments used for both feature sets")
print("- validation data remain real and untouched within each fold")


FINAL VERIFICATION PASSED
- 60 fold-level results
- 12 summary rows
- exactly 5 removed variables in Reduced feature set
- 6 models × 5 folds × 2 feature sets
- no missing evaluation metrics
- same precomputed fold assignments used for both feature sets
- validation data remain real and untouched within each fold


## Interpretation Note

This analysis is a sensitivity analysis, not a model-selection exercise. The numerical differences between the Full and Reduced feature sets should be reported descriptively.

The analysis does not establish causality and should not be described as proving that any individual predictor causes academic performance.
